# 02 — Data Preprocessing
Used Car Price Prediction and Analysis System — HOCO492 Mini Project

Demonstrates each cleaning and preprocessing decision on the actual dataset, using the exact functions in `src/preprocessing.py` and `src/feature_engineering.py` that the production training pipeline (`src/train.py`) calls.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.data_loader import load_raw_data
from src.preprocessing import clean_raw_data, build_preprocessing_pipeline, _extract_numeric
from src.feature_engineering import engineer_features
from src.config import NUMERICAL_FEATURES, CATEGORICAL_FEATURES

## 1. Load Raw Data

In [2]:
df_raw = load_raw_data()
print(df_raw.shape)
df_raw.head(3)

(8128, 13)


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0


## 2. Unit Stripping — Before / After
Text-numeric columns like `mileage` (`"23.4 kmpl"`), `engine` (`"1248 CC"`), and `max_power` (`"74 bhp"`) must have their numeric value extracted before they can be used by any ML model.

In [3]:
sample = df_raw[['mileage', 'engine', 'max_power']].head(5).copy()
sample['mileage_clean'] = _extract_numeric(sample['mileage'], r'([\d.]+)')
sample['engine_clean'] = _extract_numeric(sample['engine'], r'([\d.]+)')
sample['max_power_clean'] = _extract_numeric(sample['max_power'], r'([\d.]+)')
sample

,mileage,engine,max_power,mileage_clean,engine_clean,max_power_clean
0,23.4 kmpl,1248 CC,74 bhp,23.40,1248,74.00
1,21.14 kmpl,1498 CC,103.52 bhp,21.14,1498,103.52
2,17.7 kmpl,1497 CC,78 bhp,17.70,1497,78.00
3,23.0 kmpl,1396 CC,90 bhp,23.00,1396,90.00
4,16.1 kmpl,1298 CC,88.2 bhp,16.10,1298,88.20


## 3. Duplicate Rows

In [4]:
n_dupes = df_raw.duplicated().sum()
print(f'Exact duplicate rows before cleaning: {n_dupes}')

Exact duplicate rows before cleaning: 1202


## 4. Missing Values (Raw)

In [5]:
df_raw.isnull().sum().sort_values(ascending=False)

torque           222
seats            221
engine           221
mileage          221
max_power        215
fuel               0
km_driven          0
selling_price      0
year               0
name               0
owner              0
seller_type        0
transmission       0
dtype: int64

## 5. Full Cleaning Pipeline (`clean_raw_data`)
Applies, in order: duplicate removal -> unit stripping -> drop rows with invalid/missing target -> drop rows failing km_driven/year sanity checks -> drop the unparseable `torque` column.

In [6]:
df_clean = clean_raw_data(df_raw)
print(f'Rows: {df_raw.shape[0]} -> {df_clean.shape[0]} '
      f'({df_raw.shape[0]-df_clean.shape[0]} removed, '
      f'{(df_raw.shape[0]-df_clean.shape[0])/df_raw.shape[0]*100:.1f}%)')
print(f'Columns: {df_raw.shape[1]} -> {df_clean.shape[1]} (torque dropped)')
df_clean.dtypes

Rows: 8128 -> 6924 (1204 removed, 14.8%)
Columns: 13 -> 12 (torque dropped)


name              object
year               int64
selling_price      int64
km_driven          int64
fuel              object
seller_type       object
transmission      object
owner             object
mileage          float64
engine           float64
max_power        float64
seats            float64
dtype: object

## 6. Remaining Missing Values After Cleaning
Feature-column missing values (e.g. a few `mileage`/`engine`/`seats` entries) are intentionally NOT dropped here — they are left for `SimpleImputer` inside the modelling pipeline so the imputation statistic is learned only on the training fold, not the full dataset.

In [7]:
df_clean.isnull().sum().sort_values(ascending=False)

seats            208
mileage          208
engine           208
max_power        206
km_driven          0
selling_price      0
year               0
name               0
owner              0
transmission       0
seller_type        0
fuel               0
dtype: int64

## 7. Feature Engineering

In [8]:
df_features = engineer_features(df_clean)
df_features[['name', 'year', 'car_age', 'brand']].head(8)

,name,year,car_age,brand
0,Maruti Swift Dzire VDI,2014,10,Maruti
1,Skoda Rapid 1.5 TDI Ambition,2014,10,Skoda
2,Honda City 2017-2020 EXi,2006,18,Honda
3,Hyundai i20 Sportz Diesel,2010,14,Hyundai
4,Maruti Swift VXI BSIII,2007,17,Maruti
5,Hyundai Xcent 1.2 VTVT E Plus,2017,7,Hyundai
6,Maruti Wagon R LXI DUO BSIII,2007,17,Maruti
7,Maruti 800 DX BSII,2001,23,Maruti


### Brand Cardinality Before / After Rare-Grouping

In [9]:
raw_brand_count = df_clean['name'].astype(str).str.split().str[0].str.title().nunique()
final_brand_count = df_features['brand'].nunique()
print(f'Distinct brands before grouping: {raw_brand_count}')
print(f'Distinct brand categories after grouping rare brands into "Other": {final_brand_count}')
df_features['brand'].value_counts().tail(10)

Distinct brands before grouping: 32
Distinct brand categories after grouping rare brands into "Other": 19


brand
Volkswagen       174
Nissan            73
Skoda             70
Datsun            57
Other             56
Bmw               47
Mercedes-Benz     46
Fiat              44
Audi              33
Jeep              22
Name: count, dtype: int64

## 8. Preprocessing Pipeline (ColumnTransformer)
Numeric features: median impute -> standard scale.
Categorical features: most-frequent impute -> one-hot encode (`handle_unknown='ignore'`).

**Important:** this transformer is fit ONLY on the training split in `src/train.py` — here we fit it on the full cleaned dataset purely to demonstrate its output shape, not as part of the real training run.

In [10]:
preprocessor = build_preprocessing_pipeline()
X_demo = df_features[NUMERICAL_FEATURES + CATEGORICAL_FEATURES]
X_transformed = preprocessor.fit_transform(X_demo)
print(f'Input shape: {X_demo.shape}')
print(f'Transformed (encoded) shape: {X_transformed.shape}')
print(f'Output feature names (first 15): {list(preprocessor.get_feature_names_out()[:15])}')

Input shape: (6924, 11)
Transformed (encoded) shape: (6924, 39)
Output feature names (first 15): ['num__car_age', 'num__km_driven', 'num__mileage', 'num__engine', 'num__max_power', 'num__seats', 'cat__brand_Audi', 'cat__brand_Bmw', 'cat__brand_Chevrolet', 'cat__brand_Datsun', 'cat__brand_Fiat', 'cat__brand_Ford', 'cat__brand_Honda', 'cat__brand_Hyundai', 'cat__brand_Jeep']


## Summary
Raw dataset (8,128 rows, 13 columns) is cleaned to 6,924 rows, 12 columns (torque dropped, duplicates and invalid rows removed), then enriched with `car_age` and `brand`. The resulting feature set (6 numeric + 5 categorical = 11 model input features) is what `src/train.py` uses to train and compare all candidate models — see `03_model_training.ipynb`.